# Train y Test

Cuando utilizas el 20% para testeo y el 80% para entrenamiento el modelo se ajusta a los datos del entrenamiento apra evitar eso el autor sugiere lo siguiente para cuando hay una gran cantidad de datos:
- test 20% (no se toca datos en un ambiente real solo 1 vez se utiliza)
- entrenamiento 80% (subdividir)
  - validationset (Puede ser el 20%)-> para afinar y seleccionar modelos
  - trainsetreal (Puede ser un 60%)-> para entrenar ajustar pesos y parámetros internos

> Ojo en la compración de calidation set contra test si es muy diferente puede deverse:
> - Datos no representativos
> - Snooping Bias se utilizo información del test set durante el desarrollo.
> - Cambios de distribución datos de distintos períodos, fuente o regiones.

## Snooping Bias

Cuando el test set influye en las deccisiones dutante el desarrollo del modelo. Por ejemplo cuando se utiliza el modelo de 2 capas normal al ajustarlo puede generar generalizaciones por eso se utiliza el de 3 capas.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
import numpy as np
import pandas as pd
from zlib import crc32

In [3]:
PATH_CALIFORNIA = "/content/drive/MyDrive/machine_learning/california_housing/"
df_california_housing = pd.read_csv(f"{PATH_CALIFORNIA}housing.csv")
df_california_housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Muestreo aleatorio

In [32]:
# 1. Definimos la función correctamente con el porcentaje de corte
def shuffle_and_split_data(data, test_ratio, rng):
    # Mezclamos las posiciones (índices)
    shuffled_indices = rng.permutation(len(data))
    
    # Calculamos cuántas filas van a pruebas
    test_set_size = int(len(data) * test_ratio)
    
    # Cortamos los índices en dos grupos
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    
    # Devolvemos los datos reales partidos en Train y Test
    return data.iloc[train_indices], data.iloc[test_indices]

# 2. Creamos el generador con la semilla 42 para que no me genere cada vez aleatorios diferentes
rng = np.random.default_rng(seed=42)

# 3. Llamamos a la función pasando el DataFrame y el porcentaje (ej. 0.2 para 20% de test)
train_set, test_set = shuffle_and_split_data(df_california_housing, 0.2, rng)

In [29]:
len(train_set)

16512

In [30]:
len(test_set)

4128

In [31]:
train_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
18731,-122.41,40.55,19.0,3753.0,761.0,1952.0,738.0,3.0954,86500.0,INLAND
3002,-119.01,35.32,23.0,4870.0,965.0,2717.0,928.0,2.5960,70000.0,INLAND
19844,-119.09,36.42,17.0,877.0,219.0,966.0,218.0,2.0000,52500.0,INLAND
4939,-118.27,33.99,30.0,504.0,140.0,529.0,123.0,1.9531,100000.0,<1H OCEAN
1674,-122.27,38.04,47.0,1685.0,405.0,835.0,372.0,2.3103,134500.0,NEAR BAY


## Split por hash del identificador

Es como darle intificador a cada registro pero se utiliza cuando los datos se actualizan constantemente, en muestro caso no se va a utilizar.

CRC32 es una función matemática que recibe un dato (un número, un texto, un ID) y siempre te devuelve el mismo número entero de 32 bits.

In [34]:
help(crc32)

Help on built-in function crc32 in module zlib:

crc32(data, value=0, /)
    Compute a CRC-32 checksum of data.

      value
        Starting value of the checksum.

    The returned checksum is an integer.



In [23]:
def is_id_in_test_set(id, test_ratio):
    return crc32(np.int64(id)) < test_ratio * 2 ** 32

In [35]:
crc32(np.int64(1))

2844319735

In [36]:
crc32(np.int64(2))

654825492

In [37]:
crc32(np.int64(3))

3954038922

In [38]:
len(df_california_housing)

20640

El total de combinaciones que puede hacer es la regla de 32 bits
$$2^{32} = 2 \times 2 \times 2 \times \dots \text{ (32 veces)} = \mathbf{4,294,967,296}$$

$$\text{Umbral} = 4,294,967,296 \times 0.2 = \mathbf{858,993,459}$$

In [39]:
def split_data_with_id_hash(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_test_set(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

In [40]:
housing_with_id = df_california_housing.reset_index() # Agrega la columna "index"
train_set, test_set = split_data_with_id_hash(housing_with_id, 0.2, "index")

In [41]:
len(train_set)

16512

In [42]:
len(test_set)

4128

## Skict learn

In [43]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df_california_housing, test_size=0.2, random_state=42)

In [44]:
len(train_set)

16512

In [45]:
len(test_set)

4128

In [46]:
train_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
14196,-117.03,32.71,33.0,3126.0,627.0,2300.0,623.0,3.2596,103000.0,NEAR OCEAN
8267,-118.16,33.77,49.0,3382.0,787.0,1314.0,756.0,3.8125,382100.0,NEAR OCEAN
17445,-120.48,34.66,4.0,1897.0,331.0,915.0,336.0,4.1563,172600.0,NEAR OCEAN
14265,-117.11,32.69,36.0,1421.0,367.0,1418.0,355.0,1.9425,93400.0,NEAR OCEAN
2271,-119.80,36.78,43.0,2382.0,431.0,874.0,380.0,3.5542,96500.0,INLAND
